In [ ]:
#Importing necessary libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr
from sklearn.metrics import silhouette_score   # CORRECTION

#Reading the data
data = pd.read_csv('ifood_df.csv')
# CORRECTION
df = data.copy()
#Taking a look at the top 5 rows of the data
data.head()
data.columns
# Remove duplicates
df = df.drop_duplicates()
# Check missing values
print(df.isnull().sum())
# Fill missing Income if present
df['Income'] = df['Income'].fillna(df['Income'].median())

data = df.copy()   
data.info()
print(df.describe())
data.nunique()

data.drop(columns=['Z_CostContact','Z_Revenue'], inplace=True)

# CORRECTION
if 'Age' not in df.columns and 'Year_Birth' in df.columns:
    df['Age'] = 2025 - df['Year_Birth']
    data['Age'] = df['Age']

df['TotalPurchases'] = (df['NumWebPurchases']+ df['NumCatalogPurchases']+ df['NumStorePurchases'])
df['FamilySize'] = df['Kidhome'] + df['Teenhome']
df['AvgOrderValue'] = (df['MntTotal'] /(df['TotalPurchases'] + 1))

data = df.copy()   # CORRECTION

plt.figure(figsize=(8,5))
sns.histplot(df['Age'], bins=30, kde=True)
plt.title("Age Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df['Income'], bins=30, kde=True)
plt.title("Income Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df['MntTotal'], bins=30, kde=True)
plt.title("Total Spending Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(
    x='Income',
    y='MntTotal',
    data=df
)
plt.title("Income vs Total Spending")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(
    x='Income',
    y='TotalPurchases',
    data=df
)
plt.title("Income vs Total Purchases")
plt.show()

products = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
product_spend = df[products].mean()
plt.figure(figsize=(10,5))
product_spend.sort_values().plot(kind='bar')
plt.title("Average Spending by Product Category")
plt.ylabel("Average Spend")
plt.show()

channels = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases']
channel_avg = df[channels].mean()
plt.figure(figsize=(8,5))
channel_avg.plot(kind='bar')
plt.title("Average Purchases by Channel")
plt.show()

campaigns = ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5']
campaign_rates = df[campaigns].mean()*100
plt.figure(figsize=(8,5))
campaign_rates.plot(kind='bar')
plt.ylabel("% Accepted")
plt.title("Campaign Acceptance Rate")
plt.show()

df['Frequency'] = df['TotalPurchases']
df['Monetary'] = df['MntTotal']
df['Recency']
rfm = df[['Recency','Frequency','Monetary']]
sns.pairplot(rfm)
plt.show()

Q1 = data['MntTotal'].quantile(0.25)
Q3 = data['MntTotal'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = data[(data['MntTotal'] < lower_bound)|(data['MntTotal'] > upper_bound)]
outliers.head()
data = data[(data['MntTotal'] > lower_bound)&(data['MntTotal'] < upper_bound)]
data.describe()

cols_demographics = ['Income','Age']
cols_children = ['Kidhome', 'Teenhome']

cols_marital = ['marital_Divorced','marital_Married','marital_Single','marital_Together','marital_Widow']
cols_mnt = ['MntTotal','MntRegularProds','MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
cols_communication = ['Complain','Response','Customer_Days']
cols_campaigns = ['AcceptedCmpOverall', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
cols_source_of_purchase = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']
cols_education = ['education_2n Cycle', 'education_Basic', 'education_Graduation', 'education_Master', 'education_PhD']
corr_matrix = data[
    ['MntTotal'] +
    cols_demographics +
    cols_children
].corr()

plt.figure(figsize=(6,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    linewidths=0.5
)

plt.title('Correlation Matrix Heatmap')
plt.show()

def get_marital_status(row):

    if row['marital_Divorced'] == 1:
        return 'Divorced'
    elif row['marital_Married'] == 1:
        return 'Married'
    elif row['marital_Single'] == 1:
        return 'Single'
    elif row['marital_Together'] == 1:
        return 'Together'
    elif row['marital_Widow'] == 1:
        return 'Widow'
    else:
        return 'Unknown'

data['Marital'] = data.apply(
    get_marital_status,
    axis=1
)

plt.figure(figsize=(8, 6))

sns.barplot(
    x='Marital',
    y='MntTotal',
    data=data,
    palette='viridis'
)
plt.title('MntTotal by marital status')
plt.xlabel('Marital status')
plt.ylabel('MntTotal')

def get_relationship(row):

    if row['marital_Married'] ==1:
        return 1
    elif row['marital_Together'] == 1:
        return 1
    else:
        return 0
data['In_relationship'] = data.apply(
    get_relationship,
    axis=1
)

data.head()

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols_for_clustering = ['Income','MntTotal','In_relationship']
data_scaled = data.copy()
data_scaled[cols_for_clustering] = scaler.fit_transform(
    data[cols_for_clustering]
)
data_scaled[cols_for_clustering].describe()

from sklearn import decomposition
pca = decomposition.PCA(n_components = 2)
pca_res = pca.fit_transform(
    data_scaled[cols_for_clustering]
)
data_scaled['pc1'] = pca_res[:,0]
data_scaled['pc2'] = pca_res[:,1]

inertia = []
for k in range(1,11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    # CORRECTION
    model.fit(data_scaled[cols_for_clustering])
    inertia.append(model.inertia_)
plt.figure(figsize=(8,5))
plt.plot(
    range(1,11),
    inertia,
    marker='o'
)
plt.xlabel("Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

scores = []
for k in range(2,11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    # CORRECTION
    labels = model.fit_predict(
        data_scaled[cols_for_clustering]
    )
    scores.append(
        silhouette_score(
            data_scaled[cols_for_clustering],
            labels
        )
    )
plt.figure(figsize=(8,5))
plt.plot(
    range(2,11),
    scores,
    marker='o'
)
plt.title("Silhouette Scores")
plt.xlabel("Clusters")
plt.ylabel("Score")
plt.show()

# CORRECTION
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

data['Cluster'] = kmeans.fit_predict(
    data_scaled[cols_for_clustering]
)

data_scaled['Cluster'] = data['Cluster']

cluster_sizes = data.groupby(
    'Cluster'
)[['MntTotal']].count().reset_index()

plt.figure(figsize=(8,6))

sns.barplot(
    x='Cluster',
    y='MntTotal',
    data=cluster_sizes,
    palette='viridis'
)

plt.title('Cluster sizes')
plt.xlabel('Cluster')
plt.ylabel('MntTotal')

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='pc1',
    y='pc2',
    data=data_scaled,
    hue='Cluster',
    palette='viridis'
)

plt.title('Clustered Data Visualization')
plt.xlabel('Principal Component 1 (pc1)')
plt.ylabel('Principal Component 2 (pc2)')
plt.legend(title='Clusters')

mnt_data = data.groupby(
    'Cluster'
)[cols_mnt].mean().reset_index()

mnt_data.head()

plt.figure(figsize=(8, 6))

sns.boxplot(
    x='Cluster',
    y='Income',
    data=data,
    palette='viridis'
)

plt.title('Income by cluster')
plt.xlabel('Cluster')
plt.ylabel('Income')
plt.show()

plt.figure(figsize=(8, 6))

sns.barplot(
    x='Cluster',
    y='In_relationship',
    data=data,
    palette='viridis'
)

plt.title('In_relationship by cluster')
plt.xlabel('Cluster')
plt.ylabel('In_relationship')
plt.show()